# Chapter 15: Crawling through APIs

In [1]:
import random
import re

from bs4 import BeautifulSoup
import requests

In [2]:
ip_or_domain = "50.78.253.58"
url = f"http://ip-api.com/json/{ip_or_domain}"
resp = requests.get(url)
resp

<Response [200]>

In [3]:
resp.json()

{'status': 'success',
 'country': 'United States',
 'countryCode': 'US',
 'region': 'MA',
 'regionName': 'Massachusetts',
 'city': 'Boston',
 'zip': '02114',
 'lat': 42.3634,
 'lon': -71.0713,
 'timezone': 'America/New_York',
 'isp': 'Comcast Cable Communications',
 'org': 'Boston Park Plaza Hotel',
 'as': 'AS7922 Comcast Cable Communications, LLC',
 'query': '50.78.253.58'}

In [4]:
def get_geo_from_ip(ip_or_domain: str) -> dict:
    url = f"http://ip-api.com/json/{ip_or_domain}"
    resp = requests.get(url)
    return resp.json()

In [5]:
get_geo_from_ip("8.8.8.8")

{'status': 'success',
 'country': 'United States',
 'countryCode': 'US',
 'region': 'VA',
 'regionName': 'Virginia',
 'city': 'Ashburn',
 'zip': '20149',
 'lat': 39.03,
 'lon': -77.5,
 'timezone': 'America/New_York',
 'isp': 'Google LLC',
 'org': 'Google Public DNS',
 'as': 'AS15169 Google LLC',
 'query': '8.8.8.8'}

In [6]:
get_geo_from_ip("google.com")

{'status': 'success',
 'country': 'United States',
 'countryCode': 'US',
 'region': 'CA',
 'regionName': 'California',
 'city': 'Mountain View',
 'zip': '94043',
 'lat': 37.4225,
 'lon': -122.085,
 'timezone': 'America/Los_Angeles',
 'isp': 'Google LLC',
 'org': 'Google LLC',
 'as': 'AS15169 Google LLC',
 'query': '142.251.45.14'}

In [7]:
get_geo_from_ip("1.1.1.1")

{'status': 'success',
 'country': 'Australia',
 'countryCode': 'AU',
 'region': 'QLD',
 'regionName': 'Queensland',
 'city': 'South Brisbane',
 'zip': '4101',
 'lat': -27.4766,
 'lon': 153.0166,
 'timezone': 'Australia/Brisbane',
 'isp': 'Cloudflare, Inc',
 'org': 'APNIC and Cloudflare DNS Resolver project',
 'as': 'AS13335 Cloudflare, Inc.',
 'query': '1.1.1.1'}

In [8]:
get_geo_from_ip("msms.uztelecom.uz")

{'status': 'success',
 'country': 'Uzbekistan',
 'countryCode': 'UZ',
 'region': 'TK',
 'regionName': 'Tashkent',
 'city': 'Tashkent',
 'zip': '',
 'lat': 41.2995,
 'lon': 69.2401,
 'timezone': 'Asia/Tashkent',
 'isp': '"Uzbektelekom" Joint Stock Company',
 'org': 'Uztelecom DC',
 'as': 'AS202660 Uzbektelekom Joint Stock Company',
 'query': '92.63.206.142'}

In [9]:
headers = {"User-Agent": "Mr.Robot-Linux_Machine"}

def get_links(article_url):
    html = requests.get(f"https://en.wikipedia.org{article_url}", headers=headers)
    bs = BeautifulSoup(html.text, "html.parser")
    return bs.find("div", {"id": "bodyContent"}).find_all(
        "a", href=re.compile("^(/wiki/)((?!:).)*$")
    )


def get_history_ips(page_url):
    # Format of revision history pages is:
    # http://en.wikipedia.org/w/index.php?title=Title_in_URL&action=history
    page_url = page_url.replace("/wiki/", "")
    history_url = f"https://en.wikipedia.org/w/index.php?title={page_url}&action=history"
    print(f"History URL: {history_url}")
    bs = BeautifulSoup(
        requests.get(history_url, headers=headers).text,
        "lxml"
    )
    # finds only the links with class "mw-anonuserlink"
    # which has IP addresses instead of usernames
    ip_addresses = bs.find_all("a", {"class": "mw-anonuserlink"})
    return set(ip.get_text() for ip in ip_addresses)

In [10]:
links = get_links("/wiki/Python_(programming_language)")
print(links)

# while len(links) > 0:
#     for link in links:
#         print("-" * 20)
#         history_ips = get_history_ips(link.attrs["href"])
#         for history_ip in history_ips:
#             print(history_ip)
#     new_link = links[random.randint(0, len(links)-1)].attrs["href"]
#     links = get_links(new_link)

[]
